In [ ]:
# !pip install weaviate-client langchain tiktoken pypdf rapidocr-onnxruntime

**Connect Using Python**

In [ ]:
import weaviate
from weaviate.auth import AuthApiKey

client = weaviate.connect_to_weaviate_cloud(
    cluster_url="4mwg6uyrrnuda0x36fo0ra.c0.eu-central-1.aws.weaviate.cloud",
    auth_credentials=AuthApiKey(api_key=""),
)

print(client.is_ready())

True


**Google Colab Secrete**

In [ ]:
from google.colab import userdata

WEAVIATE_URL = userdata.get("WEAVIATE_URL")
WEAVIATE_API_KEY = userdata.get("WEAVIATE_API_KEY")
huggingfacehub_api_token = userdata.get("HUGGINGFACEHUB_API_TOKEN")

In [ ]:
from huggingface_hub import login

login(token=huggingfacehub_api_token)

In [ ]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY),
)

**Embedding model**

In [ ]:
# !pip install langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# !pip install -q langchain-community pypdf pymupdf

**Import**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

**Load PDF**

In [ ]:
loader = PyPDFLoader('/content/GenAI.pdf')
pages = loader.load()

print(len(pages))

16


**Inspect the first page**

In [ ]:
print(pages[0].page_content)

Agenda:
What is llama Index?1.
What is Lang Chain?2.
What is difference between Llama Index and Lang Chain?3.
Where we should which one?4.
Can we combine both for our application building?5.
What is Llama Index?
Llama Index is an AI data framework that helps Large Language Models (LLMs) like GPT, Gemini, Llama, Mistral, etc. connect to your own 
data.
Think of it as a bridge between your data and an LLM.
Layman Analogy:
Imagine:
LLM (ChatGPT/Gemini) = A very intelligence student.•
Your documents = Thousands of books in a library.•
LlamaIndex = The librarian.•
Llama Index supported LLMs:
OpenAI (GPT), Google Gemini, Anthropic Claude, Meta Llama, Mistral, Cohere, Ollama (Local models), Hugging Face models, NVIDIA NIMs.
Major Components of Llama Index:
Component Purpose
Readers Load data from different sources
Documents Store loaded text
Nodes Small chunks of documents
Embeddings Convert text into vectors
Index Organize data for retrieval
Retriever Find relevant chunks
Query Engine Answer

**Metadata**

In [ ]:
print(pages[0].metadata)

**Images from the PDF use PyMuPDF (fitz)**

In [ ]:
import fitz

doc = fitz.open('/content/GenAI.pdf')

for page_num in range(len(doc)):
  page = doc[page_num]
  images = page.get_images(full=True)

  print(f"Page {page_num + 1}: {len(images)} images")

Page 1: 0 images
Page 2: 0 images
Page 3: 1 images
Page 4: 0 images
Page 5: 0 images
Page 6: 2 images
Page 7: 2 images
Page 8: 3 images
Page 9: 1 images
Page 10: 2 images
Page 11: 0 images
Page 12: 0 images
Page 13: 0 images
Page 14: 0 images
Page 15: 0 images
Page 16: 0 images


This extracts embedded images separately from the text.

**Split the PDF**

In [ ]:
# !pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)

docs = text_splitter.split_documents(pages)

**Check the number of chunks**

In [ ]:
print(len(docs))

30


In [ ]:
print(docs[0].page_content)

Agenda:
What is llama Index?1.
What is Lang Chain?2.
What is difference between Llama Index and Lang Chain?3.
Where we should which one?4.
Can we combine both for our application building?5.
What is Llama Index?
Llama Index is an AI data framework that helps Large Language Models (LLMs) like GPT, Gemini, Llama, Mistral, etc. connect to your own 
data.
Think of it as a bridge between your data and an LLM.
Layman Analogy:
Imagine:
LLM (ChatGPT/Gemini) = A very intelligence student.•
Your documents = Thousands of books in a library.•
LlamaIndex = The librarian.•
Llama Index supported LLMs:
OpenAI (GPT), Google Gemini, Anthropic Claude, Meta Llama, Mistral, Cohere, Ollama (Local models), Hugging Face models, NVIDIA NIMs.
Major Components of Llama Index:
Component Purpose
Readers Load data from different sources
Documents Store loaded text
Nodes Small chunks of documents
Embeddings Convert text into vectors
Index Organize data for retrieval
Retriever Find relevant chunks


In [ ]:
print(docs[0].metadata)

In [ ]:
docs

In [ ]:
# !pip install langchain-weaviate

In [ ]:
from langchain_weaviate import WeaviateVectorStore

vector_db = WeaviateVectorStore.from_documents(
    documents=docs,
    embedding=embedding_model,
    client=client,
    index_name="RAGDocuments",
)

In [ ]:
print(vector_db.similarity_search("what is rag?", k=3)[0].page_content)

In [ ]:

print(vector_db.similarity_search("what is rag?", k=3)[1].page_content)

In [ ]:

print(vector_db.similarity_search("what is rag?", k=3)[2].page_content)

In [ ]:
print(
    vector_db.similarity_search(
        "what is attention?", k=3)
    )

In [ ]:
from langchain_core.prompts import PromptTemplate # Changed from ChatPromptTemplate

template = """
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.

Question:
{question}

Context:
{context}

Answer:
"""
prompt = PromptTemplate.from_template(template) # Changed from ChatPromptTemplate

In [ ]:
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
# !pip install -U langchain-huggingface huggingface_hub

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="conversational", # Corrected task back to 'conversational' as required by the model
    huggingfacehub_api_token=huggingfacehub_api_token,
    temperature=0.7,
    max_new_tokens=180
)

In [ ]:
model = llm # Directly use the HuggingFaceEndpoint as the model

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
output_parser = StrOutputParser()

In [ ]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | output_parser
)

In [ ]:
try:
    response = rag_chain.invoke(
        "What is RAG system?"
    )

    print(response)
except Exception as e:
    print(f"An error occurred during rag_chain invocation: {type(e).__name__}: {e}")

An error occurred during rag_chain invocation: ValueError: Model mistralai/Mistral-7B-Instruct-v0.2 is not supported for task text-generation and provider featherless-ai. Supported task: conversational.
